# Парсинг драгметаллов

Для драг. металлов и валюты:
- engine: `currency`
- market: `selt`
- board: `cets`

Тикеры драгметаллов и валюты можно найти здесь: `https://iss.moex.com/iss/engines/currency/markets/selt/boards/CETS/securities`




In [4]:
from enum import Enum, IntEnum
from aiomoex import get_board_candles
import asyncio
import aiohttp
from datetime import datetime, timedelta
import pandas as pd


class Engines(Enum):
    """https://iss.moex.com/iss/engines"""

    STOCK = "stock"  # Фондовый рынок и рынок депозитов
    STATE = "state"  # Рынок ГЦБ (размещение)
    CURRENCY = "currency"  # Валютный рынок
    FUTURES = "futures"  # stockСрочный рынок
    COMMODITY = "commodity"  # Товарный рынок
    INTERVENTIONS = "interventions"  # Товарные интервенции
    OFFBOARD = "offboard"  # ОТС-система
    AGR = "agro"  # Агро
    OTC = "otc"  # ОТС с ЦК
    QUOTES = "quotes"  # Квоты
    MONEY = "money"  # Денежный рынок


class Markets(Enum):
    """https://iss.moex.com/iss/engines/<engine>/markets"""

    # engine=currency
    OTCINDICES = "otcindices"  # Внебиржевые индексы
    SELT = "selt"  # Биржевые сделки с ЦК
    FUTURES = "futures"  # Поставочные фьючерсы
    INDEX = "index"  # Валютный фиксинг
    OTC = "otc"  # Внебиржевой
    
    # engine=futures
    FORTS = "forts"  # Фьючерсы
    OPTIONS = "options"  # Опционы
    FORTSIQS = "fortsiqs"  # Фьючерсы IQS
    OPTIONSIQS = "optionsiqs"  # Опционы IQS
    MAIN = "main"  # Срочные инструменты


class Boards(Enum):
    """https://iss.moex.com/iss/engines/<engine>/markets/<market>/boards"""

    # engine=currency, market=selt
    TQBR = "TQBR"  # Фондовый рынок
    AUCB = "AUCB"  # Аукцион ЦБР - адрес.
    CETS = "CETS"  # Системные сделки - безадрес.
    CNGD = "CNGD"  # Внесистемные сделки- адрес.
    CURR = "CURR"  # Дневная сессия
    FIXN = "FIXN"  # Фиксинг внесистемный- адрес.
    FIXS = "FIXS"  # Фиксинг системный - безадрес.
    LICU = "LICU"  # Внесистемные сделки урегулирования - безадрес.
    SDBP = "SDBP"  # Крупные сделки - безадрес.
    SPEC = "SPEC"  # Поставка - безадресные
    WAPN = "WAPN"  # Внесистемные средневзвешенные - адрес.
    WAPS = "WAPS"  # Системные средневзвешенные - безадрес.

    # engine=futures, market=forts
    RFUD = "RFUD"  # Фьючерсы

class IntervalEnum(IntEnum):
    MINUTE = 1
    TEN_MINUTES = 10
    HOUR = 60
    DAY = 24
    WEEK = 7
    MONTH = 31


# объявим аннотацию для удобства
StockData = list[dict[str, str | int | float]]


async def fetch_ticker_data(
    session: aiohttp.ClientSession,
    ticker: str,
    interval: IntervalEnum,
    start_date: str,
    end_date: str,
    board: Boards,
    engine: Engines,
    market: Markets,
) -> dict[str, StockData]:
    """Функция получает данные о торгах по заданному тикеру с *start_date* по *end_date* с интервалом *interval*, возвращая словарь, где ключом является тикер, а значением - данные

    Args:
        session (aiohttp.ClientSession): aiottp сессия для отпаравки запросов
        ticker (str): Имя тикера
        interval (IntervalEnum): Одно из доступных значений для интервала времени
        start_date (str): Начальная дата в формате yyyy-mm-dd
        end_date (str): Конечная дата в формате yyyy-mm-dd

    Returns:
        dict[str, list[dict[str, str | int | float]]]: Словарь, где ключ - тикер, а значение - данные, например {'SBER': sber_data}
    """
    try:
        print(f"Запрашиваем данные для {ticker}...")
        # получаем данные по переданному тикеру за указанный период
        res = await get_board_candles(
            session,
            ticker,
            interval,
            start_date,
            end_date,
            board=board.value,
            market=market.value,
            engine=engine.value,
        )
        print(f"Получено {len(res)} записей для {ticker}")
        return {ticker: res}
    except Exception as e:
        print(f"Ошибка парсинга. Не удалось получить данные для {ticker}: {e}")
        print(f"Тип ошибки: {type(e).__name__}")
        return {ticker: []}


async def get_moex_data(
    tickers: list[str],
    start_date: datetime,
    end_date: datetime = datetime.now(),
    interval: IntervalEnum = IntervalEnum.DAY,
    engine: Engines = Engines.CURRENCY,
    market: Markets = Markets.SELT,
    board: Boards = Boards.CETS,
) -> dict[str, StockData]:
    if interval not in IntervalEnum:
        raise ValueError(f"Неверный интервал. Допустимые значения: {IntervalEnum}")

    end_date_formatted = end_date.strftime("%Y-%m-%d")
    start_date_formatted = start_date.strftime("%Y-%m-%d")
    
    print(f"Запрашиваем данные для тикеров: {tickers}")
    print(f"Период: {start_date_formatted} - {end_date_formatted}")
    print(f"Параметры: engine={engine.value}, market={market.value}, board={board.value}")

    # Увеличиваем таймауты для MOEX API
    timeout = aiohttp.ClientTimeout(
        connect=30,      # время подключения
        sock_read=60,   # время чтения данных
        total=120       # общий таймаут
    )
    
    async with aiohttp.ClientSession(timeout=timeout) as session:
        # собираем корутины в список
        coros = [
            fetch_ticker_data(
                session,
                ticker,
                interval,
                start_date_formatted,
                end_date_formatted,
                board,
                engine,
                market,
            )
            for ticker in tickers
        ]

        print("Отправляем запросы к MOEX API...")
        # 'собираем' результаты корутин - непосредственно парсинг
        stock_data = await asyncio.gather(*coros)
        print("Получены ответы от MOEX API")

    # разворачиваем список словарей в один словарь, например: [ {'SBER': sber_data}, {'GAZP': gazp_data} ] -> { 'SBER': sber_data, 'GAZP': gazp_data }
    stock_data = {
        ticker: data for element in stock_data for ticker, data in element.items()
    }
    
    # Выводим информацию о полученных данных
    for ticker, data in stock_data.items():
        print(f"Тикер {ticker}: получено {len(data)} записей")
        if data:
            print(f"Первая запись: {data[0]}")
    
    return stock_data

In [ ]:
from dataclasses import dataclass


@dataclass
class Asset:
    """Общий класс для активов с нужными параметрами для получения данных с MOEX"""
    engine: Engines
    market: Markets
    board: Boards
    ticker: str

    async def get_candles(self, start_date: datetime, end_date: datetime, interval: IntervalEnum) -> pd.DataFrame:
        data =  await get_moex_data(
            tickers=[self.ticker],
            start_date=start_date,
            end_date=end_date,
            interval=interval,
            engine=self.engine,
            market=self.market,
            board=self.board,
        )
        df = pd.DataFrame(data[self.ticker])
        df["ticker"] = self.ticker
        return df

In [6]:
gold = Asset(
    engine=Engines.CURRENCY,
    market=Markets.SELT,
    board=Boards.CETS,
    ticker="GLDRUB_TOM",
)
display(await gold.get_candles(start_date=datetime.now() - timedelta(days=180), end_date=datetime.now(), interval=IntervalEnum.DAY))

Запрашиваем данные для тикеров: ['GLDRUB_TOM']
Период: 2025-05-01 - 2025-10-28
Параметры: engine=currency, market=selt, board=CETS
Отправляем запросы к MOEX API...
Запрашиваем данные для GLDRUB_TOM...
Получено 125 записей для GLDRUB_TOM
Получены ответы от MOEX API
Тикер GLDRUB_TOM: получено 125 записей
Первая запись: {'open': 8560, 'close': 8599.8, 'high': 8690, 'low': 8556.3, 'value': 676326652.9, 'volume': 78401, 'begin': '2025-05-02 00:00:00', 'end': '2025-05-02 23:59:59'}


,open,close,high,low,value,volume,begin,end,ticker
0,8560.0,8599.8,8690.0,8556.3,6.763267e+08,78401,2025-05-02 00:00:00,2025-05-02 23:59:59,GLDRUB_TOM
1,8672.5,8675.0,8684.9,8599.9,1.570787e+09,181738,2025-05-05 00:00:00,2025-05-05 23:59:59,GLDRUB_TOM
2,8756.5,8802.0,8824.9,8741.0,2.238773e+09,254910,2025-05-06 00:00:00,2025-05-06 23:59:59,GLDRUB_TOM
3,8825.0,8853.2,8890.0,8748.0,2.241656e+09,254278,2025-05-07 00:00:00,2025-05-07 23:59:59,GLDRUB_TOM
4,8770.0,8773.1,8861.0,8759.8,2.894047e+08,32847,2025-05-08 00:00:00,2025-05-08 23:59:59,GLDRUB_TOM
...,...,...,...,...,...,...,...,...,...
120,11251.0,10880.0,11289.9,10660.0,7.543267e+09,687506,2025-10-21 00:00:00,2025-10-21 23:59:59,GLDRUB_TOM
121,11000.0,10529.0,11037.0,10490.0,9.297243e+09,869267,2025-10-22 00:00:00,2025-10-22 23:59:59,GLDRUB_TOM
122,10750.0,10850.0,10850.0,10609.6,5.016714e+09,468023,2025-10-23 00:00:00,2025-10-23 23:59:59,GLDRUB_TOM
123,10689.0,10554.9,10719.7,10434.2,5.510741e+09,521940,2025-10-24 00:00:00,2025-10-24 23:59:59,GLDRUB_TOM


In [7]:
silver = Asset(
    engine=Engines.CURRENCY,
    market=Markets.SELT,
    board=Boards.CETS,
    ticker="SLVRUB_TOM",
)
display(await silver.get_candles(start_date=datetime.now() - timedelta(days=180), end_date=datetime.now(), interval=IntervalEnum.DAY))

Запрашиваем данные для тикеров: ['SLVRUB_TOM']
Период: 2025-05-01 - 2025-10-28
Параметры: engine=currency, market=selt, board=CETS
Отправляем запросы к MOEX API...
Запрашиваем данные для SLVRUB_TOM...
Получено 125 записей для SLVRUB_TOM
Получены ответы от MOEX API
Тикер SLVRUB_TOM: получено 125 записей
Первая запись: {'open': 120.21, 'close': 122.33, 'high': 123.29, 'low': 120.21, 'value': 13317770, 'volume': 108800, 'begin': '2025-05-02 00:00:00', 'end': '2025-05-02 23:59:59'}


,open,close,high,low,value,volume,begin,end,ticker
0,120.21,122.33,123.29,120.21,13317770,108800,2025-05-02 00:00:00,2025-05-02 23:59:59,SLVRUB_TOM
1,122.35,121.18,124.00,121.07,22147602,179800,2025-05-05 00:00:00,2025-05-05 23:59:59,SLVRUB_TOM
2,121.66,122.50,124.00,121.66,27198687,220900,2025-05-06 00:00:00,2025-05-06 23:59:59,SLVRUB_TOM
3,122.49,123.21,125.00,121.20,26538636,215200,2025-05-07 00:00:00,2025-05-07 23:59:59,SLVRUB_TOM
4,123.21,122.53,125.51,121.91,16131188,130500,2025-05-08 00:00:00,2025-05-08 23:59:59,SLVRUB_TOM
...,...,...,...,...,...,...,...,...,...
120,183.25,166.80,187.50,165.35,314513388,1792000,2025-10-21 00:00:00,2025-10-21 23:59:59,SLVRUB_TOM
121,166.00,152.00,173.70,147.00,385098061,2461500,2025-10-22 00:00:00,2025-10-22 23:59:59,SLVRUB_TOM
122,154.41,173.00,174.30,154.40,345496743,2075500,2025-10-23 00:00:00,2025-10-23 23:59:59,SLVRUB_TOM
123,169.79,159.00,169.90,157.57,179007448,1092000,2025-10-24 00:00:00,2025-10-24 23:59:59,SLVRUB_TOM


In [8]:
platinum = Asset(
    engine=Engines.CURRENCY,
    market=Markets.SELT,
    board=Boards.CETS,
    ticker="PLTRUB_TOM",
)
display(await platinum.get_candles(start_date=datetime.now() - timedelta(days=180), end_date=datetime.now(), interval=IntervalEnum.DAY))

Запрашиваем данные для тикеров: ['PLTRUB_TOM']
Период: 2025-05-01 - 2025-10-28
Параметры: engine=currency, market=selt, board=CETS
Отправляем запросы к MOEX API...
Запрашиваем данные для PLTRUB_TOM...
Получено 125 записей для PLTRUB_TOM
Получены ответы от MOEX API
Тикер PLTRUB_TOM: получено 125 записей
Первая запись: {'open': 2575, 'close': 2573.98, 'high': 2585.99, 'low': 2550, 'value': 4940590.35, 'volume': 1928, 'begin': '2025-05-02 00:00:00', 'end': '2025-05-02 23:59:59'}


,open,close,high,low,value,volume,begin,end,ticker
0,2575.00,2573.98,2585.99,2550.00,4940590.35,1928,2025-05-02 00:00:00,2025-05-02 23:59:59,PLTRUB_TOM
1,2573.00,2539.98,2573.00,2525.01,11879892.62,4675,2025-05-05 00:00:00,2025-05-05 23:59:59,PLTRUB_TOM
2,2539.98,2564.00,2569.99,2539.98,7953406.83,3118,2025-05-06 00:00:00,2025-05-06 23:59:59,PLTRUB_TOM
3,2564.00,2577.01,2586.89,2564.00,1434233.10,557,2025-05-07 00:00:00,2025-05-07 23:59:59,PLTRUB_TOM
4,2585.00,2589.99,2606.00,2564.04,3736200.30,1453,2025-05-08 00:00:00,2025-05-08 23:59:59,PLTRUB_TOM
...,...,...,...,...,...,...,...,...,...
120,4162.00,3958.96,4174.00,3849.18,74578677.45,18424,2025-10-21 00:00:00,2025-10-21 23:59:59,PLTRUB_TOM
121,3962.01,3950.00,4049.97,3900.01,41873972.49,10531,2025-10-22 00:00:00,2025-10-22 23:59:59,PLTRUB_TOM
122,4000.40,4146.99,4158.00,4000.40,23557896.15,5765,2025-10-23 00:00:00,2025-10-23 23:59:59,PLTRUB_TOM
123,4100.00,4054.97,4140.57,3980.02,25440644.31,6315,2025-10-24 00:00:00,2025-10-24 23:59:59,PLTRUB_TOM


In [9]:
palladium = Asset(
    engine=Engines.CURRENCY,
    market=Markets.SELT,
    board=Boards.CETS,
    ticker="PLDRUB_TOM",
)
display(await palladium.get_candles(start_date=datetime.now() - timedelta(days=180), end_date=datetime.now(), interval=IntervalEnum.DAY))

Запрашиваем данные для тикеров: ['PLDRUB_TOM']
Период: 2025-05-01 - 2025-10-28
Параметры: engine=currency, market=selt, board=CETS
Отправляем запросы к MOEX API...
Запрашиваем данные для PLDRUB_TOM...
Получено 125 записей для PLDRUB_TOM
Получены ответы от MOEX API
Тикер PLDRUB_TOM: получено 125 записей
Первая запись: {'open': 2490, 'close': 2519.88, 'high': 2527.98, 'low': 2475.01, 'value': 1130835.37, 'volume': 451, 'begin': '2025-05-02 00:00:00', 'end': '2025-05-02 23:59:59'}


,open,close,high,low,value,volume,begin,end,ticker
0,2490.00,2519.88,2527.98,2475.01,1130835.37,451,2025-05-02 00:00:00,2025-05-02 23:59:59,PLDRUB_TOM
1,2519.88,2503.00,2531.38,2490.01,9683523.56,3849,2025-05-05 00:00:00,2025-05-05 23:59:59,PLDRUB_TOM
2,2496.24,2540.00,2540.00,2496.24,1959760.70,781,2025-05-06 00:00:00,2025-05-06 23:59:59,PLDRUB_TOM
3,2540.00,2570.00,2570.00,2505.02,2277552.98,896,2025-05-07 00:00:00,2025-05-07 23:59:59,PLDRUB_TOM
4,2570.00,2564.00,2570.00,2528.01,2906534.81,1142,2025-05-08 00:00:00,2025-05-08 23:59:59,PLDRUB_TOM
...,...,...,...,...,...,...,...,...,...
120,3699.98,3495.00,3699.98,3386.00,61717150.98,17566,2025-10-21 00:00:00,2025-10-21 23:59:59,PLDRUB_TOM
121,3550.00,3399.55,3600.00,3360.02,15463606.43,4451,2025-10-22 00:00:00,2025-10-22 23:59:59,PLDRUB_TOM
122,3405.35,3529.99,3580.00,3405.35,22717918.59,6471,2025-10-23 00:00:00,2025-10-23 23:59:59,PLDRUB_TOM
123,3500.00,3473.00,3500.00,3337.81,26745679.04,7872,2025-10-24 00:00:00,2025-10-24 23:59:59,PLDRUB_TOM


# Нефть

In [10]:
# MOEX

# На MOEX нет индекса цены нефти, есть только фьючерсы.
# Список фьючерсов можно глянуть здесь:
# https://iss.moex.com/iss/engines/futures/markets/forts/boards/rfud/securities

BRENT_FUTURES = [
    "BRF6", # 1.26
    "BRG6", # 2.26
    "BRZ5", # 12.25
]

for future_name in BRENT_FUTURES:
    print('Future:', future_name)
    brent_future = Asset(
        engine=Engines.FUTURES,
        market=Markets.FORTS,
        board=Boards.RFUD,
        ticker=future_name,
    )
    display(await brent_future.get_candles(start_date=datetime.now() - timedelta(days=180), end_date=datetime.now(), interval=IntervalEnum.DAY))


Future: BRF6
Запрашиваем данные для тикеров: ['BRF6']
Период: 2025-05-01 - 2025-10-28
Параметры: engine=futures, market=forts, board=RFUD
Отправляем запросы к MOEX API...
Запрашиваем данные для BRF6...
Получено 126 записей для BRF6
Получены ответы от MOEX API
Тикер BRF6: получено 126 записей
Первая запись: {'open': 65, 'close': 64.5, 'high': 65.66, 'low': 64.39, 'value': 0, 'volume': 23, 'begin': '2025-05-02 00:00:00', 'end': '2025-05-02 23:59:59'}


,open,close,high,low,value,volume,begin,end,ticker
0,65.00,64.50,65.66,64.39,0,23,2025-05-02 00:00:00,2025-05-02 23:59:59,BRF6
1,64.41,63.52,64.41,62.88,0,46,2025-05-05 00:00:00,2025-05-05 23:59:59,BRF6
2,64.60,64.90,64.90,64.54,0,3,2025-05-06 00:00:00,2025-05-06 19:30:22,BRF6
3,65.18,65.00,66.02,65.00,0,3,2025-05-07 00:00:00,2025-05-07 23:59:59,BRF6
4,65.39,65.39,65.39,65.39,0,1,2025-05-08 00:00:00,2025-05-08 23:59:59,BRF6
...,...,...,...,...,...,...,...,...,...
121,60.93,62.20,62.20,60.60,0,2834,2025-10-22 00:00:00,2025-10-22 23:59:59,BRF6
122,62.05,64.48,64.62,61.80,0,4062,2025-10-23 00:00:00,2025-10-23 23:59:59,BRF6
123,64.55,64.92,65.05,64.02,0,1844,2025-10-24 00:00:00,2025-10-24 23:59:59,BRF6
124,65.02,64.55,65.03,63.70,0,1974,2025-10-27 00:00:00,2025-10-27 23:59:59,BRF6


Future: BRG6
Запрашиваем данные для тикеров: ['BRG6']
Период: 2025-05-01 - 2025-10-28
Параметры: engine=futures, market=forts, board=RFUD
Отправляем запросы к MOEX API...
Запрашиваем данные для BRG6...
Получено 117 записей для BRG6
Получены ответы от MOEX API
Тикер BRG6: получено 117 записей
Первая запись: {'open': 64.86, 'close': 64.41, 'high': 64.86, 'low': 64.41, 'value': 0, 'volume': 3, 'begin': '2025-05-02 00:00:00', 'end': '2025-05-02 23:59:59'}


,open,close,high,low,value,volume,begin,end,ticker
0,64.86,64.41,64.86,64.41,0,3,2025-05-02 00:00:00,2025-05-02 23:59:59,BRG6
1,63.98,63.38,63.98,63.38,0,3,2025-05-05 00:00:00,2025-05-05 23:59:59,BRG6
2,68.59,67.99,68.59,67.99,0,4,2025-05-12 00:00:00,2025-05-12 23:59:59,BRG6
3,67.33,67.99,67.99,67.33,0,2,2025-05-13 00:00:00,2025-05-13 23:59:59,BRG6
4,68.33,68.33,68.33,68.33,0,2,2025-05-14 00:00:00,2025-05-14 23:59:59,BRG6
...,...,...,...,...,...,...,...,...,...
112,60.47,60.83,61.35,60.23,0,355,2025-10-21 00:00:00,2025-10-21 23:59:59,BRG6
113,61.11,62.11,62.20,60.67,0,473,2025-10-22 00:00:00,2025-10-22 23:59:59,BRG6
114,62.08,64.07,64.26,61.81,0,1297,2025-10-23 00:00:00,2025-10-23 23:59:59,BRG6
115,64.22,64.51,65.00,63.66,0,419,2025-10-24 00:00:00,2025-10-24 23:59:59,BRG6


Future: BRZ5
Запрашиваем данные для тикеров: ['BRZ5']
Период: 2025-05-01 - 2025-10-28
Параметры: engine=futures, market=forts, board=RFUD
Отправляем запросы к MOEX API...
Запрашиваем данные для BRZ5...
Получено 126 записей для BRZ5
Получены ответы от MOEX API
Тикер BRZ5: получено 126 записей
Первая запись: {'open': 64.8, 'close': 63.3, 'high': 65.22, 'low': 63.28, 'value': 0, 'volume': 37, 'begin': '2025-05-02 00:00:00', 'end': '2025-05-02 23:59:59'}


,open,close,high,low,value,volume,begin,end,ticker
0,64.80,63.30,65.22,63.28,0,37,2025-05-02 00:00:00,2025-05-02 23:59:59,BRZ5
1,63.94,63.03,63.94,62.20,0,20,2025-05-05 00:00:00,2025-05-05 23:59:59,BRZ5
2,64.97,64.90,64.97,63.24,0,9,2025-05-06 00:00:00,2025-05-06 19:30:22,BRZ5
3,65.19,64.72,65.20,64.04,0,8,2025-05-07 00:00:00,2025-05-07 23:59:59,BRZ5
4,64.50,64.71,64.73,64.20,0,8,2025-05-08 00:00:00,2025-05-08 23:59:59,BRZ5
...,...,...,...,...,...,...,...,...,...
121,61.11,62.50,62.50,60.80,0,56942,2025-10-22 00:00:00,2025-10-22 23:59:59,BRZ5
122,62.39,65.16,65.36,61.97,0,121286,2025-10-23 00:00:00,2025-10-23 23:59:59,BRZ5
123,65.20,65.68,65.83,64.67,0,80535,2025-10-24 00:00:00,2025-10-24 23:59:59,BRZ5
124,65.70,65.16,65.72,64.23,0,103206,2025-10-27 00:00:00,2025-10-27 23:59:59,BRZ5


In [ ]:

def fetch_brent_eia(start=START, end=END) -> pd.DataFrame:
    api_key = os.environ["EIA_API_KEY"]  # обязан быть задан
    url = "https://api.eia.gov/v2/petroleum/pri/spt/data/"
    params = {
        "api_key": api_key,
        "frequency": "daily",
        "data[0]": "value",
        "facets[series][]": "RBRTE",  # Europe Brent Spot FOB
        "start": start.isoformat(),
        "end": end.isoformat(),
        "sort[0][column]": "period",
        "sort[0][direction]": "asc",
    }
    r = S.get(url, params=params, timeout=30); r.raise_for_status()
    data = r.json()["response"]["data"]
    df = pd.DataFrame(data)[["period","value"]].rename(columns={"period":"date","value":"brent_usd_per_bbl"})
    df["date"] = pd.to_datetime(df["date"]).dt.date
    df["source_brent"] = "eia-v2:RBRTE"
    return df

def fetch_urals_te(start=START, end=END) -> pd.DataFrame:
    client = os.environ["TE_CLIENT"]
    secret = os.environ["TE_SECRET"]
    url = "https://api.tradingeconomics.com/commodities/urals%20oil"
    params = {
        "d1": start.isoformat(),
        "d2": end.isoformat(),
        "format": "json",
        "c": f"{client}:{secret}",
    }
    r = S.get(url, params=params, timeout=30); r.raise_for_status()
    js = r.json()
    # ожидаемые поля: 'Date' (ISO) и 'Price' (float)
    rows = [(pd.to_datetime(x["Date"]).date(), float(x["Price"])) for x in js if "Date" in x and "Price" in x]
    df = pd.DataFrame(rows, columns=["date","urals_usd_per_bbl"]).drop_duplicates("date").sort_values("date")
    df["source_urals"] = "tradingeconomics:UralsOil"
    return df

In [30]:
from investiny import historical_data

historical_data(investing_id=6408, from_date="09/01/2022", to_date="10/01/2022") # Returns AAPL historical data as JSON (without date)

ConnectionError: Request to Investing.com API failed with error code: 403.

# Парсинг валют

Цены USD и EUR берем с сайта Центробанка РФ. Цены юаня можно взять на MOEX.

Курсы валют на определенный день: `https://www.cbr.ru/scripts/XML_daily.asp?date_req=28.10.2025` (выдает ближайшую дату, если по указанной дате нет данных)
Курсы валют за определенный период: `https://www.cbr.ru/scripts/XML_dynamic.asp?date_req1=01/04/2022&date_req2=28/10/2025&VAL_NM_RQ=R01235`


In [11]:
# MOEX

# С MOEX удалось спарсить только цены юаня. Евро и доллар не торгуются на мосбирже.
# Вот, например, график пары USD/RUB. Последняя отметка 11.06.2024:
# https://www.moex.com/ru/issue/USD000UTSTOM/CETS
cnyrub = Asset(
    engine=Engines.CURRENCY,
    market=Markets.SELT,
    board=Boards.CETS,
    ticker="CNYRUB_TOM",
)
display(await cnyrub.get_candles(start_date=datetime.now() - timedelta(days=180), end_date=datetime.now(), interval=IntervalEnum.DAY))

Запрашиваем данные для тикеров: ['CNYRUB_TOM']
Период: 2025-05-01 - 2025-10-28
Параметры: engine=currency, market=selt, board=CETS
Отправляем запросы к MOEX API...
Запрашиваем данные для CNYRUB_TOM...


Получено 125 записей для CNYRUB_TOM
Получены ответы от MOEX API
Тикер CNYRUB_TOM: получено 125 записей
Первая запись: {'open': 11.2305, 'close': 11.41, 'high': 11.478, 'low': 11.2305, 'value': 20100890186.5, 'volume': 1762922000, 'begin': '2025-05-02 00:00:00', 'end': '2025-05-02 23:59:59'}


,open,close,high,low,value,volume,begin,end,ticker
0,11.2305,11.4100,11.4780,11.2305,2.010089e+10,1762922000,2025-05-02 00:00:00,2025-05-02 23:59:59,CNYRUB_TOM
1,11.4740,11.2080,11.4900,11.1630,7.664625e+10,6799211000,2025-05-05 00:00:00,2025-05-05 23:59:59,CNYRUB_TOM
2,11.1980,11.2350,11.2660,11.1425,6.961663e+10,6206679000,2025-05-06 00:00:00,2025-05-06 23:59:59,CNYRUB_TOM
3,11.2290,11.1950,11.2450,11.1235,7.835026e+10,7004193000,2025-05-07 00:00:00,2025-05-07 23:59:59,CNYRUB_TOM
4,11.2205,11.2375,11.3675,11.2010,1.737354e+10,1538351000,2025-05-08 00:00:00,2025-05-08 23:59:59,CNYRUB_TOM
...,...,...,...,...,...,...,...,...,...
120,11.3300,11.4080,11.4570,11.2850,1.133614e+11,9971020000,2025-10-21 00:00:00,2025-10-21 23:59:59,CNYRUB_TOM
121,11.4250,11.3740,11.5265,11.3465,1.515163e+11,13239364000,2025-10-22 00:00:00,2025-10-22 23:59:59,CNYRUB_TOM
122,11.4520,11.3770,11.4520,11.3250,1.078361e+11,9481156000,2025-10-23 00:00:00,2025-10-23 23:59:59,CNYRUB_TOM
123,11.3885,11.1470,11.4070,11.1110,1.732666e+11,15354477000,2025-10-24 00:00:00,2025-10-24 23:59:59,CNYRUB_TOM


In [31]:
# API Центробанка
# Здесь можно получить только цену за определенный день. Объема торгов нет, т.к. USD и EUR не торгуются на мосбирже.

from datetime import datetime
from typing import Dict
import requests
import pandas as pd
import xml.etree.ElementTree as ET
from enum import Enum

# MOEX_DAILY_URL_TEMPLATE = "https://www.cbr.ru/scripts/XML_daily.asp?date_req={date}"
MOEX_DYNAMIC_URL_TEMPLATE = "https://cbr.ru/scripts/XML_dynamic.asp?date_req1={start}&date_req2={end}&VAL_NM_RQ={ticker}"


class MoexTicker(Enum):
    USDRUB = "R01235"
    EURRUB = "R01239"
    CNYRUB = "R01375"

def fetch_cbr_series(ticker: MoexTicker, start: datetime, end: datetime) -> pd.DataFrame:
    """
    Fetch dynamic series for a currency by its VAL_NM_RQ (e.g., R01235) between start and end
    and return list of (date_iso, value_rub_per_1_unit).
    """
    url = MOEX_DYNAMIC_URL_TEMPLATE.format(ticker=ticker.value, start=start.strftime("%d/%m/%Y"), end=end.strftime("%d/%m/%Y"))
    # CBR responds in windows-1251; use bytes and decode as needed
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    # Parse XML
    root = ET.fromstring(r.content)
    out: list[tuple[str, float]] = []
    for rec in root.findall("Record"):
        d_mdy = rec.attrib.get("Date")  # format: DD.MM.YYYY
        if not d_mdy:
            continue
        d_iso = datetime.strptime(d_mdy, "%d.%m.%Y").date().strftime("%Y-%m-%d")
        nominal_el = rec.find("Nominal")
        value_el = rec.find("Value")
        if nominal_el is None or value_el is None:
            continue
        try:
            nominal = int(nominal_el.text.strip())
        except Exception:
            nominal = 1
        # Decimal with comma
        value_txt = value_el.text.strip().replace(",", ".")
        rub_per_nominal = float(value_txt)
        rub_per_one = rub_per_nominal / nominal if nominal else rub_per_nominal
        out.append((d_iso, rub_per_one))
    # Drop duplicate dates keeping the last
    # (CBR should not duplicate, but be safe)
    out_dict: Dict[str, float] = {}
    for d, v in out:
        out_dict[d] = v

    out_sorted = sorted(out_dict.items(), key=lambda t: t[0])

    out_sorted = [
        (
            value,
            date + " 00:00:00",
            date + " 23:59:59",
            ticker.name
        ) for date, value in out_sorted]
    return pd.DataFrame(out_sorted, columns=["value", "date_start", "date_end", "ticker"])

display(fetch_cbr_series(MoexTicker.USDRUB, start=datetime(2022, 4, 1), end=datetime.now()))
display(fetch_cbr_series(MoexTicker.EURRUB, start=datetime(2022, 4, 1), end=datetime.now()))
display(fetch_cbr_series(MoexTicker.CNYRUB, start=datetime(2022, 4, 1), end=datetime.now()))

,value,date_start,date_end,ticker
0,83.4097,2022-04-01 00:00:00,2022-04-01 23:59:59,USDRUB
1,83.4285,2022-04-02 00:00:00,2022-04-02 23:59:59,USDRUB
2,83.5932,2022-04-05 00:00:00,2022-04-05 23:59:59,USDRUB
3,83.3520,2022-04-06 00:00:00,2022-04-06 23:59:59,USDRUB
4,82.5962,2022-04-07 00:00:00,2022-04-07 23:59:59,USDRUB
...,...,...,...,...
883,81.3475,2025-10-22 00:00:00,2025-10-22 23:59:59,USDRUB
884,81.6549,2025-10-23 00:00:00,2025-10-23 23:59:59,USDRUB
885,81.2690,2025-10-24 00:00:00,2025-10-24 23:59:59,USDRUB
886,80.9713,2025-10-25 00:00:00,2025-10-25 23:59:59,USDRUB


,value,date_start,date_end,ticker
0,92.4930,2022-04-01 00:00:00,2022-04-01 23:59:59,EURRUB
1,92.1468,2022-04-02 00:00:00,2022-04-02 23:59:59,EURRUB
2,92.3872,2022-04-05 00:00:00,2022-04-05 23:59:59,EURRUB
3,91.7289,2022-04-06 00:00:00,2022-04-06 23:59:59,EURRUB
4,90.5998,2022-04-07 00:00:00,2022-04-07 23:59:59,EURRUB
...,...,...,...,...
883,94.6656,2025-10-22 00:00:00,2025-10-22 23:59:59,EURRUB
884,94.7543,2025-10-23 00:00:00,2025-10-23 23:59:59,EURRUB
885,94.3889,2025-10-24 00:00:00,2025-10-24 23:59:59,EURRUB
886,94.0820,2025-10-25 00:00:00,2025-10-25 23:59:59,EURRUB


,value,date_start,date_end,ticker
0,13.1569,2022-04-01 00:00:00,2022-04-01 23:59:59,CNYRUB
1,13.1111,2022-04-02 00:00:00,2022-04-02 23:59:59,CNYRUB
2,13.1372,2022-04-05 00:00:00,2022-04-05 23:59:59,CNYRUB
3,13.0993,2022-04-06 00:00:00,2022-04-06 23:59:59,CNYRUB
4,12.9838,2022-04-07 00:00:00,2022-04-07 23:59:59,CNYRUB
...,...,...,...,...
883,11.3645,2025-10-22 00:00:00,2025-10-22 23:59:59,CNYRUB
884,11.4541,2025-10-23 00:00:00,2025-10-23 23:59:59,CNYRUB
885,11.3733,2025-10-24 00:00:00,2025-10-24 23:59:59,CNYRUB
886,11.3079,2025-10-25 00:00:00,2025-10-25 23:59:59,CNYRUB
